# 02 GDC 전수 조사와 코호트 정의 (후보 0)
TCGA-LUAD·LUSC의 오믹스 파일 목록(크기 포함)을 GDC에서 조회하고(다운로드 없음), 환자 × 데이터 층 표를 만든다.
**코호트 = 원발 종양 RNA(STAR counts)가 있는 환자.** 기존 v3 메틸화 코호트(750명) 포함 여부도 표시한다.
- 커널: `C:\Users\Admin\EXAONEPath\exaonepath-venv` (인터넷 필요)
- 출력: `C:\Users\Admin\EXAONEPath_data\v5\02_gdc_inventory\{gdc_files.csv, availability.csv, cohort.csv}`

In [1]:
# 1. 경로와 GDC 조회 함수
from pathlib import Path
import requests, pandas as pd

V4C = Path(r"C:\Users\Admin\EXAONEPath_data\v4\02_cohort\cohort.csv")   # 기존 메틸화 코호트 750명
OUT = Path(r"C:\Users\Admin\EXAONEPath_data\v5\02_gdc_inventory")
OUT.mkdir(parents=True, exist_ok=True)
PROJ = ["TCGA-LUAD", "TCGA-LUSC"]

def gdc(filters, fields):
    r = requests.post("https://api.gdc.cancer.gov/files", timeout=600,
                      json={"filters": filters, "fields": ",".join(fields), "format": "JSON", "size": 100000})
    r.raise_for_status()
    d = r.json()["data"]
    assert len(d["hits"]) == d["pagination"]["total"], "페이지 누락"
    return d["hits"]

In [2]:
# 2. 오믹스 파일 목록 (슬라이드 제외)
LAYER = {"Gene Expression Quantification": "RNA", "miRNA Expression Quantification": "miRNA",
         "Masked Somatic Mutation": "MAF", "Gene Level Copy Number": "CNV",
         "Protein Expression Quantification": "RPPA", "Methylation Beta Value": "Meth"}
hits = gdc({"op": "and", "content": [{"op": "in", "content": {"field": "cases.project.project_id", "value": PROJ}},
                                     {"op": "in", "content": {"field": "data_type", "value": list(LAYER)}}]},
           ["file_id", "file_name", "file_size", "md5sum", "data_type", "platform", "analysis.workflow_type",
            "cases.submitter_id", "cases.project.project_id", "cases.samples.submitter_id", "cases.samples.sample_type"])

f = pd.DataFrame([(h["file_id"], h["file_name"], h["file_size"], h["md5sum"], h["data_type"], h.get("platform"),
                   (h.get("analysis") or {}).get("workflow_type"), c["project"]["project_id"][5:], c["submitter_id"],
                   s.get("submitter_id"), s.get("sample_type"))
                  for h in hits for c in h["cases"] for s in c.get("samples", [])],
                 columns=["file_id", "file_name", "file_size", "md5", "data_type", "platform", "workflow",
                          "cancer", "patient_id", "sample_id", "sample_type"])
f["layer"] = f.data_type.map(LAYER)
f.loc[f.layer == "Meth", "layer"] = f.platform.str.contains("450", na=False).map({True: "Meth450", False: "Meth_other"})
f.to_csv(OUT / "gdc_files.csv", index=False, encoding="utf-8-sig")
print(f.groupby(["layer", "workflow"], dropna=False).size())

layer       workflow                                            
CNV         ABSOLUTE LiftOver                                       1005
            ASCAT2                                                  2116
            ASCAT3                                                  1986
            AscatNGS                                                 148
MAF         Aliquot Ensemble Somatic Variant Merging and Masking    2334
Meth450     SeSAMe Methylation Beta Estimation                       919
Meth_other  SeSAMe Methylation Beta Estimation                       364
RNA         STAR - Counts                                           1163
RPPA        NaN                                                      693
miRNA       BCGSC miRNA Profiling                                   1100
dtype: int64


In [3]:
# 3. 환자 × 데이터 층 (원발 종양) → 코호트
pt = f[f.sample_type == "Primary Tumor"]
av = (pd.crosstab([pt.patient_id, pt.cancer], pt.layer) > 0).reset_index()
av["v3_meth_cohort"] = av.patient_id.isin(pd.read_csv(V4C).patient_id)
av.to_csv(OUT / "availability.csv", index=False, encoding="utf-8-sig")

co = av[av.RNA].drop(columns="RNA")
co.to_csv(OUT / "cohort.csv", index=False, encoding="utf-8-sig")
print("전체 환자", av.groupby("cancer").size().to_dict(), "→ 코호트(RNA)", co.groupby("cancer").size().to_dict())
co.groupby("cancer").sum(numeric_only=True).astype(int)

전체 환자 {'LUAD': 584, 'LUSC': 504} → 코호트(RNA) {'LUAD': 517, 'LUSC': 501}


layer,CNV,MAF,Meth450,Meth_other,RPPA,miRNA,v3_meth_cohort
cancer,,,,,,,
LUAD,516,507,455,115,361,510,404
LUSC,501,487,370,130,325,485,347


In [4]:
# 4. 다운로드 예상 용량 (코호트 원발 종양, 층별)
d = pt[pt.patient_id.isin(co.patient_id)]
(d.drop_duplicates("file_id").groupby("layer").agg(files=("file_id", "size"), GB=("file_size", lambda x: round(x.sum() / 2**30, 2))))

,files,GB
layer,,
CNV,3110,9.97
MAF,1098,0.10
Meth450,840,10.26
Meth_other,248,1.37
RNA,1051,4.15
RPPA,686,0.01
miRNA,1001,0.05
